In [ ]:
# CREATING THE FEATURE VARIABLES
# Bitcoin trading applet that detects breakout candles. T
# That is the candles that yield a 'Six percent gain in Six days'

# In this file our goal is to: 
# take in the btc_1d_2018-2025-ORIGINAL.csv and generate feature variables. 
# the new feature variables will be added to make a new csv to train a model on.

# The variables for: btc_1d_2018-2025-ORIGINAL.csv:
# Open time, Open, High, Low, Close, Volume, Close time, Quote asset volume, Number of trades, 
# Taker buy base asset volume, Taker buy quote asset volume

# We want to add these feature variable columns: 
# trigger: a binary 0,1. 1 signals the candle that could breakout 6 percent in 6 days	
# ret_1d: The return from open to close(could be minus, lost money)
# ret_3d: The return from three days close to close (could be open to open?)	
# ret_5d: The return from five day close to close (could be open to open?)	
# vol_5d: The volume from five days ago cumulative to current candle	
# rsi_14: The relative strecgth index after 14 days
# atr_14: The average true range after 14 days

# From here we will run a correlation matrix and continue dropping variables in the next .ipynb file.

In [12]:
# Import libraries
import pandas as pd
import numpy as np
print('successfully imported libraries needed')

successfully imported libraries needed


In [16]:
# Read in the full csv created from the Binance API
# Basic data cleaning

df = pd.read_csv('btc_1d_2018-2025-ORIGINAL.csv')
#print(df.shape) # (2847, 12)
#df.info()

# change col names to lower for simplicity
# Rename 'open time' to 'date'
df.columns = df.columns.str.strip().str.lower()
df = df.rename(columns={'open time': 'date'})
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# output a mini df with number of na values per column
#print(df.isna().sum()) # There are no na vaues in any columns

for col in ['open', 'high', 'low', 'close', 'volume']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=['date', 'open', 'high', 'low', 'close'])
df = df.drop_duplicates(subset=['date'])
df = df.sort_values('date').reset_index(drop=True)

# Force the chronological sequence if rows were removed
df = df.reset_index(drop=True)

print(df.shape)
df.head()

(2847, 12)


,date,open,high,low,close,volume,close time,quote asset volume,number of trades,taker buy base asset volume,taker buy quote asset volume,trigger
0,2018-01-01 00:00:00+00:00,13715.65,13818.55,12750.00,13380.00,8609.915844,2018-01-01 23:59:59.999000 UTC,114799747.4,105595,3961.938946,5.280975e+07,0
1,2018-01-02 00:00:00+00:00,13382.16,15473.49,12890.02,14675.11,20078.092110,2018-01-02 23:59:59.999000 UTC,279717107.4,177728,11346.326740,1.580801e+08,0
2,2018-01-03 00:00:00+00:00,14690.00,15307.56,14150.00,14919.51,15905.667640,2018-01-03 23:59:59.999000 UTC,236116868.3,162787,8994.953566,1.335873e+08,0
3,2018-01-04 00:00:00+00:00,14919.51,15280.00,13918.04,15059.54,21329.649570,2018-01-04 23:59:59.999000 UTC,312781583.8,170310,12680.812950,1.861168e+08,0
4,2018-01-05 00:00:00+00:00,15059.56,17176.24,14600.00,16960.39,23251.491130,2018-01-05 23:59:59.999000 UTC,369321956.5,192969,13346.622290,2.118299e+08,0


In [17]:
# ADDING FEATURE VARIABLES -- TRY 2 (LOOKS ACCURATE)

# =========================
# 1. LOAD DATA (if needed)
# =========================
df = pd.read_csv('btc_1d_2018-2025-ORIGINAL.csv')

df.columns = df.columns.str.strip().str.lower()
df = df.rename(columns={'open time': 'date'})

df['date'] = pd.to_datetime(df['date'], errors='coerce')

for col in ['open', 'high', 'low', 'close', 'volume']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=['date', 'open', 'high', 'low', 'close'])
df = df.drop_duplicates(subset=['date'])
df = df.sort_values('date').reset_index(drop=True)


# =========================
# 2. LABEL (trigger)
# =========================
horizon = 6
target = 0.06

future_max_high = df['high'][::-1].rolling(horizon).max()[::-1]

df['trigger'] = ((future_max_high - df['close']) / df['close'] >= target).astype(int)


# =========================
# 3. RETURNS FEATURES
# =========================
df['ret_1d'] = df['close'].pct_change(1)
df['ret_3d'] = df['close'].pct_change(3)
df['ret_5d'] = df['close'].pct_change(5)


# =========================
# 4. VOLUME FEATURES
# =========================
df['vol_5d'] = df['volume'].rolling(5).sum()
df['vol_5d_avg'] = df['volume'].rolling(5).mean()
df['vol_ratio'] = df['volume'] / df['vol_5d_avg']


# =========================
# 5. RSI (14)
# =========================
def compute_rsi(series, period=14):
    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df['rsi_14'] = compute_rsi(df['close'], 14)


# =========================
# 6. ATR (14)
# =========================
high_low = df['high'] - df['low']
high_close = abs(df['high'] - df['close'].shift(1))
low_close = abs(df['low'] - df['close'].shift(1))

true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)

df['atr_14'] = true_range.rolling(14).mean()


# =========================
# 7. CLEANUP
# =========================
df = df.dropna().reset_index(drop=True)


# =========================
# 8. EXPORT
# =========================
df.to_csv('df_full_features_added-V3.csv', index=False)

print("Saved: df_full_features_added-V3.csv")
print(df.shape)
df.head()

Saved: df_full_features_added-V3.csv
(2833, 20)


,date,open,high,low,close,volume,close time,quote asset volume,number of trades,taker buy base asset volume,taker buy quote asset volume,trigger,ret_1d,ret_3d,ret_5d,vol_5d,vol_5d_avg,vol_ratio,rsi_14,atr_14
0,2018-01-15 00:00:00+00:00,13477.98,14249.99,13147.79,13539.93,14652.09471,2018-01-15 23:59:59.999000 UTC,200614554.5,145362,7431.063272,101748315.6,0,0.004819,-0.014562,-0.091712,93863.38346,18772.676692,0.780501,50.775896,1860.680000
1,2018-01-16 00:00:00+00:00,13500.00,13542.93,9035.00,10900.00,63401.16918,2018-01-16 23:59:59.999000 UTC,722150828.1,418243,30353.401720,346777069.9,1,-0.194974,-0.232935,-0.176661,123709.82889,24741.965778,2.562495,33.799160,1998.141429
2,2018-01-17 00:00:00+00:00,10899.99,11680.99,9037.94,10988.79,72331.79665,2018-01-17 23:59:59.999000 UTC,744154180.1,517293,35412.798140,365047291.4,1,0.008146,-0.184505,-0.200234,179624.46340,35924.892680,2.013417,32.903015,2104.247857
3,2018-01-18 00:00:00+00:00,10972.59,11878.82,10435.33,10961.97,48464.43494,2018-01-18 23:59:59.999000 UTC,541289614.1,393112,23814.382850,266091805.7,1,-0.002441,-0.190397,-0.228574,215867.38981,43173.477962,1.122551,32.000018,2110.071429
4,2018-01-19 00:00:00+00:00,10960.00,11795.00,10360.00,11474.98,34129.37542,2018-01-19 23:59:59.999000 UTC,382773499.2,273271,16871.416360,189259676.9,1,0.046799,0.052750,-0.148424,232978.87090,46595.774180,0.732456,22.557335,2028.554286
